<a href="https://colab.research.google.com/github/YonggunJung/Lotto/blob/main/1243LottoGPT_Astra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


# ---------- 설정 ----------
CSV_FILE = "/content/drive/MyDrive/Colab Notebooks/로또/data/lotto2.csv"

# Colab에 직접 업로드했다면:
# CSV_FILE = "/content/lotto2.csv"

# 실행할 때 업로드 창을 띄우려면:
# CSV_FILE = ""

OUTPUT_FILE = "lotto_new_candidates.csv"

ORDER_MODE = "auto"          # 이번 파일의 앞 1085개 역순 구간 보정
WINDOWS = (5, 10, 20, 50, 100)

TEST_ROUNDS = 200            # 모델 선택에서 제외할 마지막 평가 구간
CV_SPLITS = 3
CV_ROUNDS = 100              # 검증 구간 하나의 회차 수

N_TICKETS = 10               # 생성할 조합 개수
DIVERSITY = 0.5              # 후보 간 번호 집중 완화. 0이면 적용하지 않음
SEED = 42                   # 바꾸면 생성되는 후보가 달라질 수 있음

BASE = 6 / 45
METHODS = ("균등", "누적빈도", "로지스틱", "부스팅")


def load_rows(path):
    """파일 검증 및 회차 순서 보정."""
    df = pd.read_csv(
        path,
        header=0,
        encoding="utf-8-sig",
        skip_blank_lines=False,
    )

    if df.columns.astype(str).str.strip().tolist() != [
        "1", "2", "3", "4", "5", "6"
    ]:
        raise ValueError(
            "첫 줄이 1,2,3,4,5,6인 lotto2.csv를 사용하세요."
        )

    a = df.apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(dtype=float)

    valid = (
        np.isfinite(a).all(axis=1)
        & (a == np.floor(a)).all(axis=1)
        & ((a >= 1) & (a <= 45)).all(axis=1)
    )

    valid &= np.array(
        [len(set(row)) == 6 for row in a],
        dtype=bool,
    )

    if not valid.all():
        lines = (np.flatnonzero(~valid) + 2).tolist()
        raise ValueError(
            f"CSV {lines}번째 행의 빈칸/정수/범위/중복을 확인하세요."
        )

    minimum = (
        max(WINDOWS)
        + TEST_ROUNDS
        + CV_SPLITS * CV_ROUNDS
        + 300
    )

    if len(a) < minimum:
        raise ValueError(
            f"현재 검증 설정에는 최소 {minimum}회차가 필요합니다."
        )

    rows = np.sort(a.astype(int), axis=1)

    first = (10, 23, 29, 33, 37, 40)
    second = (9, 13, 21, 25, 32, 42)
    keys = list(map(tuple, rows))

    if ORDER_MODE == "newest_first":
        rows = rows[::-1].copy()

    elif ORDER_MODE == "auto":
        if (
            len(rows) > 1085
            and keys[0] == (4, 7, 17, 18, 38, 44)
            and keys[1084] == first
            and keys[1083] == second
            and keys[1085] == (11, 16, 25, 27, 35, 36)
        ):
            rows = np.concatenate([
                rows[:1085][::-1],
                rows[1085:],
            ])
            print(
                "앞 1085개 데이터를 뒤집어 회차 순서를 보정했습니다."
            )

        elif keys[:2] == [first, second]:
            pass

        elif keys[-2:] == [second, first]:
            rows = rows[::-1].copy()

        else:
            raise ValueError(
                "행 순서를 확인하세요. "
                "정순 파일은 ORDER_MODE='oldest_first'."
            )

    elif ORDER_MODE != "oldest_first":
        raise ValueError(
            "ORDER_MODE는 auto / oldest_first / newest_first "
            "중 하나입니다."
        )

    repeats = sum(
        count - 1
        for count in Counter(map(tuple, rows)).values()
    )

    print(
        f"데이터 {len(rows):,}회차 / "
        f"과거 조합 반복 {repeats}개"
    )
    print("최신 데이터:", rows[-1].tolist())

    return rows


def make_features(rows):
    """t회차 입력에는 t-1회차까지의 정보만 사용합니다."""
    n = len(rows)

    hot = np.zeros((n, 45), dtype=np.uint8)
    hot[np.arange(n)[:, None], rows - 1] = 1

    cumulative = np.vstack([
        np.zeros((1, 45)),
        np.cumsum(hot, axis=0),
    ])

    # 번호를 크기 관계 없이 구분하기 위한 정보
    identity = np.eye(45)

    last_seen = np.full(45, -1)
    recent_average = np.full(45, BASE)

    blocks = []
    frequencies = []
    start = max(WINDOWS)

    for t in range(n + 1):
        if t >= start:
            # 최근 5/10/20/50/100회 출현 비율
            rates = [
                (cumulative[t] - cumulative[t - w]) / w
                for w in WINDOWS
            ]

            # 누적 빈도에 균등 사전값을 더해 극단값 완화
            frequency = (
                cumulative[t] + 100 * BASE
            ) / (t + 100)

            # 마지막 출현 이후 지나간 회차 수
            gap = np.minimum(
                t - 1 - last_seen,
                100,
            ) / 100

            # 최근 10회와 50회 출현 비율 차이
            trend = rates[1] - rates[3]

            block = np.column_stack([
                hot[t - 1],
                *rates,
                frequency,
                gap,
                recent_average,
                trend,
                identity,
            ])

            blocks.append(block)
            frequencies.append(frequency)

        if t < n:
            # 현재 결과는 입력을 만든 다음에 반영
            last_seen[hot[t] == 1] = t
            recent_average = (
                0.9 * recent_average
                + 0.1 * hot[t]
            )

    features = np.asarray(blocks, dtype=np.float32)
    frequencies = np.asarray(frequencies, dtype=float)

    return (
        features[:-1],
        hot[start:],
        frequencies[:-1],
        features[-1:],
        frequencies[-1:],
    )


def predict_method(
    name,
    train_x,
    train_y,
    predict_x,
    frequencies,
):
    if name == "균등":
        return np.full(
            (len(predict_x), 45),
            BASE,
        )

    if name == "누적빈도":
        return frequencies.copy()

    if name == "로지스틱":
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.1,
                max_iter=1000,
                solver="lbfgs",
            ),
        )

    elif name == "부스팅":
        model = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=120,
            max_leaf_nodes=7,
            min_samples_leaf=200,
            l2_regularization=10,
            early_stopping=False,
            random_state=SEED,
        )

    else:
        raise ValueError(f"알 수 없는 방식: {name}")

    # 먼저 회차 단위로 분리한 뒤 번호별 표본으로 펼칩니다.
    # 같은 회차의 번호가 학습/검증 양쪽에 섞이지 않습니다.
    model.fit(
        train_x.reshape(-1, train_x.shape[-1]),
        train_y.ravel(),
    )

    p = model.predict_proba(
        predict_x.reshape(-1, predict_x.shape[-1])
    )[:, 1]

    return p.reshape(len(predict_x), 45)


def evaluate(y, p, seed):
    if (
        p.shape != y.shape
        or not np.isfinite(p).all()
        or ((p < 0) | (p > 1)).any()
    ):
        raise ValueError("모델 출력 형태 또는 확률 범위 오류")

    # Brier: 예측값과 실제 0/1 결과의 제곱 오차
    # 낮을수록 좋습니다.
    brier = float(np.mean((p - y) ** 2))

    # 점수가 같으면 무작위로 순서를 정해
    # 특정 번호가 항상 선택되는 것을 막습니다.
    rng = np.random.default_rng(seed)
    tie_break = rng.uniform(0, 1e-12, p.shape)

    ranking = np.argsort(
        -(p + tie_break),
        axis=1,
    )[:, :6]

    hits = np.take_along_axis(
        y,
        ranking,
        axis=1,
    ).sum(axis=1)

    return brier, float(hits.mean())


def compare_methods(x, y, frequency):
    end = len(x) - TEST_ROUNDS

    cv = TimeSeriesSplit(
        n_splits=CV_SPLITS,
        test_size=CV_ROUNDS,
    )

    cv_scores = {
        name: []
        for name in METHODS
    }

    for fold, (train, valid) in enumerate(
        cv.split(x[:end]),
        1,
    ):
        print(
            f"검증 {fold}/{CV_SPLITS}: "
            f"학습 {len(train)}회 / 검증 {len(valid)}회",
            flush=True,
        )

        for name in METHODS:
            p = predict_method(
                name,
                x[train],
                y[train],
                x[valid],
                frequency[valid],
            )

            cv_scores[name].append(
                evaluate(y[valid], p, SEED + fold)
            )

    mean_brier = {
        name: np.mean([
            score[0]
            for score in cv_scores[name]
        ])
        for name in METHODS
    }

    # 마지막 평가 구간의 결과를 보기 전에 선택
    selected = min(
        METHODS,
        key=lambda name: mean_brier[name],
    )

    print("검증 오차로 선택한 방식:", selected)

    # 마지막 구간은 별도 평가에만 사용
    records = []

    for name in METHODS:
        p = predict_method(
            name,
            x[:end],
            y[:end],
            x[end:],
            frequency[end:],
        )

        test_brier, test_hits = evaluate(
            y[end:],
            p,
            SEED + 100,
        )

        records.append({
            "방식": name,
            "검증_Brier": mean_brier[name],
            "평가_Brier": test_brier,
            "평가_평균일치": test_hits,
        })

    report = pd.DataFrame(records)

    print(
        f"\n마지막 {TEST_ROUNDS}회 평가 "
        "(방식 선택에 사용하지 않음):"
    )

    print(
        report.to_string(
            index=False,
            float_format=lambda value: f"{value:.6f}",
        )
    )

    print(
        "균등 무작위로 6개를 고를 때 "
        "평균 일치 개수의 이론값: 0.8개"
    )
    print(
        "이 표는 번호별 예측값과 상위 6개 평가이며 "
        "최종 10개 조합의 적중률은 아닙니다."
    )

    if selected == "균등":
        print(
            "검증에서 다른 방식의 개선이 확인되지 않아 "
            "번호를 균등하게 취급합니다."
        )
    else:
        print(
            "검증 오차가 가장 낮은 방식을 사용합니다. "
            "향후 우수성을 보장하지는 않습니다."
        )

    return selected, report


def generate_tickets(probabilities, history):
    if (
        N_TICKETS < 1
        or not np.isfinite(DIVERSITY)
        or DIVERSITY < 0
    ):
        raise ValueError(
            "N_TICKETS는 1 이상, DIVERSITY는 0 이상이어야 합니다."
        )

    p = np.asarray(probabilities, dtype=float)

    if (
        p.shape != (45,)
        or not np.isfinite(p).all()
        or ((p < 0) | (p > 1)).any()
    ):
        raise ValueError("후보 생성용 번호 점수가 잘못되었습니다.")

    rng = np.random.default_rng(SEED)
    usage = np.zeros(45)

    chosen = []
    seen = set()

    for _ in range(max(10000, N_TICKETS * 100)):
        # 모든 45개 번호를 후보로 유지합니다.
        # 이미 많이 사용한 번호의 가중치를 조금 낮춥니다.
        weights = (
            np.maximum(p, 1e-8)
            / (1 + DIVERSITY * usage)
        )

        indices = rng.choice(
            45,
            size=6,
            replace=False,
            p=weights / weights.sum(),
        )

        combo = tuple(
            sorted((indices + 1).tolist())
        )

        # 과거와 6개 전부 같거나 출력 후보와 같으면 제외
        if combo in history or combo in seen:
            continue

        chosen.append(combo)
        seen.add(combo)
        usage[indices] += 1

        if len(chosen) == N_TICKETS:
            return chosen

    raise RuntimeError(
        "조건을 만족하는 후보가 부족합니다. 출력 개수를 줄이세요."
    )


def main(csv_file=None, output_file=None):
    path = CSV_FILE if csv_file is None else str(csv_file)

    if not path:
        from google.colab import files

        uploaded = files.upload()

        if len(uploaded) != 1:
            raise ValueError("lotto2.csv 하나만 선택하세요.")

        path = next(iter(uploaded))

    if (
        path.startswith("/content/drive/")
        and not Path("/content/drive/MyDrive").exists()
    ):
        from google.colab import drive
        drive.mount("/content/drive")

    if not Path(path).is_file():
        raise FileNotFoundError(
            f"CSV_FILE 경로를 확인하세요: {path}"
        )

    output = Path(
        OUTPUT_FILE if output_file is None else output_file
    )

    if Path(path).resolve() == output.resolve():
        raise ValueError("입력 CSV와 출력 파일은 달라야 합니다.")

    rows = load_rows(path)

    x, y, frequency, next_x, next_freq = make_features(rows)

    with threadpool_limits(limits=2):
        selected, report = compare_methods(
            x,
            y,
            frequency,
        )

        # 선택한 방식으로 전체 과거 데이터를 사용해 다음 회차 계산
        p = predict_method(
            selected,
            x,
            y,
            next_x,
            next_freq,
        )[0]

    history = {
        tuple(row)
        for row in rows
    }

    tickets = generate_tickets(p, history)

    if (
        len(set(tickets)) != len(tickets)
        or any(ticket in history for ticket in tickets)
    ):
        raise RuntimeError("저장 전 완전 중복 검사 실패")

    result = pd.DataFrame(
        tickets,
        columns=[f"번호{i}" for i in range(1, 7)],
    )

    result.insert(
        0,
        "후보",
        np.arange(1, len(result) + 1),
    )

    print("\n새 후보 조합:")
    print(result.to_string(index=False))

    output.parent.mkdir(parents=True, exist_ok=True)

    result.to_csv(
        output,
        index=False,
        encoding="utf-8-sig",
    )

    print("저장 완료:", output.resolve())
    print(
        "후보의 분산과 과거 조합 제외는 "
        "당첨 확률 향상을 보장하지 않습니다."
    )

    return result, report


if __name__ == "__main__":
    main()

Mounted at /content/drive
앞 1085개 데이터를 뒤집어 회차 순서를 보정했습니다.
데이터 1,242회차 / 과거 조합 반복 0개
최신 데이터: [2, 4, 10, 16, 31, 41]
검증 1/3: 학습 642회 / 검증 100회
검증 2/3: 학습 742회 / 검증 100회
검증 3/3: 학습 842회 / 검증 100회
검증 오차로 선택한 방식: 균등

마지막 200회 평가 (방식 선택에 사용하지 않음):
  방식  검증_Brier  평가_Brier  평가_평균일치
  균등  0.115556  0.115556 0.760000
누적빈도  0.115737  0.115643 0.785000
로지스틱  0.115861  0.115882 0.735000
 부스팅  0.115766  0.115620 0.755000
균등 무작위로 6개를 고를 때 평균 일치 개수의 이론값: 0.8개
이 표는 번호별 예측값과 상위 6개 평가이며 최종 10개 조합의 적중률은 아닙니다.
검증에서 다른 방식의 개선이 확인되지 않아 번호를 균등하게 취급합니다.

새 후보 조합:
 후보  번호1  번호2  번호3  번호4  번호5  번호6
  1    5   20   32   35   39   44
  2    6   17   21   34   36   42
  3    3   10   20   25   29   37
  4   16   28   34   38   41   44
  5    2    7    9   21   30   34
  6    9   15   16   21   33   44
  7    6   11   19   22   30   38
  8   13   14   18   31   37   38
  9    1    6    9   29   31   36
 10    5    7   22   26   32   36
저장 완료: /content/lotto_new_candidates.csv
후보의 분산과 과거 조합 제외는 당첨 확률 향상을 보장하지 않습니다.


In [2]:
[1     5   20   32   35   39   44]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2608894741.py, line 1)